## Recommendation Approach and Optimisation

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import pdist
from IPython.display import display

In [ ]:
# Load clustered member profiles
df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/clustered_member_profiles.csv')

# Separate feature vector sub-spaces
tfidf_cols = [c for c in df.columns if c.startswith('tfidf_')]
sector_cols = [c for c in df.columns if c.startswith('sec_')]
seniority_cols = [c for c in df.columns if c.startswith('sen_')]
behavior_cols = ['norm_attendance', 'norm_seniority']

print(f"Total members: {len(df)}")
print(f"TF-IDF features: {len(tfidf_cols)}")
print(f"Sector features: {len(sector_cols)}")
print(f"Seniority features: {len(seniority_cols)}")

In [ ]:
# Build a flexible recommender class
class CyNamRecommender:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.tfidf_cols = [c for c in data.columns if c.startswith('tfidf_')]
        self.sector_cols = [c for c in data.columns if c.startswith('sec_')]
        self.seniority_cols = [c for c in data.columns if c.startswith('sen_')]

        # Precompute sub-matrices
        self.X_tfidf = self.data[self.tfidf_cols].values
        self.X_sector = self.data[self.sector_cols].values
        self.X_sen = self.data['norm_seniority'].values
        self.X_att = self.data['norm_attendance'].values

    def recommend(self, query_email, mode='homophily', top_n=5, w_interest=0.5, w_sector=0.3, w_sen=0.2, serendipity_prob=0.0):
        if query_email not in self.data['email'].values:
            return f"Error: Email {query_email} not found."

        target_idx = self.data[self.data['email'] == query_email].index[0]
        target_cluster = self.data.loc[target_idx, 'cluster']
        target_sen = self.X_sen[target_idx]

        # 1. Cosine similarity for semantic event interests
        sim_interest = cosine_similarity(
            self.X_tfidf[target_idx].reshape(1, -1), self.X_tfidf).flatten()

        # 2. Sector match / complementarity
        sim_sector = cosine_similarity(
            self.X_sector[target_idx].reshape(1, -1), self.X_sector).flatten()

        # 3. Seniority logic depending on mode
        # 'homophily': close in seniority (peer-to-peer)
        # 'complementarity' (mentorship/guidance): senior mentor to junior, or vice versa
        if mode == 'homophily':
            sen_score = 1.0 - np.abs(self.X_sen - target_sen)
        elif mode == 'mentorship':
            # Target wants someone more senior (or if target is exec, wants high-potential student/junior)
            if target_sen <= 0.5:  # student/junior/mid
                # favors higher seniority
                sen_score = np.maximum(0, self.X_sen - target_sen)
            else:  # exec/senior
                # favors junior/mentees
                sen_score = np.maximum(0, target_sen - self.X_sen)
        elif mode == 'cross_sector':
            # Invert sector similarity to encourage cross-domain exploration
            sim_sector = 1.0 - sim_sector
            sen_score = 1.0 - np.abs(self.X_sen - target_sen)
        else:  # default balanced
            sen_score = 1.0 - np.abs(self.X_sen - target_sen)

        # Composite score
        final_scores = (w_interest * sim_interest) + \
            (w_sector * sim_sector) + (w_sen * sen_score)

        # Exclude self and members from the same exact organization (to promote ecosystem networking)
        target_org = str(
            self.data.loc[target_idx, 'organization']).strip().lower()
        if target_org and target_org != 'nan':
            same_org = self.data['organization'].fillna(
                '').str.strip().str.lower() == target_org
            final_scores[same_org] = -1.0
        else:
            final_scores[target_idx] = -1.0

        # Optional Serendipity: inject exploration from other clusters
        if serendipity_prob > 0:
            diff_cluster_mask = (self.data['cluster'] != target_cluster).values
            boost = np.random.uniform(0, serendipity_prob, size=len(
                final_scores)) * diff_cluster_mask
            final_scores += boost

        # Rank top N
        top_indices = np.argsort(final_scores)[::-1][:top_n]

        results = []
        for idx in top_indices:
            row = self.data.iloc[idx]
            results.append({
                'Name': row['full_name'],
                'Job Title': row['job_title'] if pd.notna(row['job_title']) else 'Unknown',
                'Organization': row['organization'] if pd.notna(row['organization']) else 'Unknown',
                'Sector': row['clean_sector'],
                'Seniority Level': row['seniority_level_name'],
                'Cluster': row['cluster'],
                'Events Attended': int(row['event_attendance_count']),
                'Match Score': round(float(final_scores[idx]), 4),
                'Interest Sim': round(float(sim_interest[idx]), 3),
                'Sector Sim': round(float(sim_sector[idx]), 3)
            })

        return pd.DataFrame(results)

In [ ]:
rec_engine = CyNamRecommender(df)

# Test with a known active member
sample_emails = df[df['job_title'].notna() & (
    df['event_attendance_count'] >= 3)]['email'].head(10).tolist()
print("Sample active emails:", sample_emails[:3])

In [ ]:
# Let's inspect member 1: a student or junior practitioner
student_members = df[(df['seniority_level_name'] ==
                      'Entry/Student') & (df['event_attendance_count'] >= 2)]
print("Students count with >=2 events:", len(student_members))
student_test_email = student_members.iloc[0]['email']
print("Testing Student:", student_test_email)

In [ ]:
# Let's inspect member 2: a senior / executive founder
exec_members = df[(df['seniority_level_name'] ==
                   'Executive/Leadership') & (df['event_attendance_count'] >= 4)]
print("Execs count with >=4 events:", len(exec_members))
exec_test_email = exec_members.iloc[0]['email']
print("Testing Exec:", exec_test_email)

In [ ]:
print("\nTEST 1: Student Mentorship Recommendations")
print("Target:", df[df['email'] == student_test_email][[
      'full_name', 'job_title', 'organization', 'clean_sector']].to_dict(orient='records'))
student_recs = rec_engine.recommend(
    student_test_email, mode='mentorship', top_n=5)
display(student_recs[['Name', 'Job Title', 'Organization',
        'Seniority Level', 'Match Score', 'Interest Sim']])

print("\nTEST 2: Executive Peer Homophily Recommendations")
print("Target:", df[df['email'] == exec_test_email][[
      'full_name', 'job_title', 'organization', 'clean_sector']].to_dict(orient='records'))
exec_recs = rec_engine.recommend(exec_test_email, mode='homophily', top_n=5)
display(exec_recs[['Name', 'Job Title', 'Organization',
        'Seniority Level', 'Match Score', 'Interest Sim']])

print("\nTEST 3: Cross-Sector Innovation Recommendations")
cross_recs = rec_engine.recommend(
    exec_test_email, mode='cross_sector', top_n=5)
display(cross_recs[['Name', 'Job Title', 'Organization',
        'Sector', 'Match Score', 'Sector Sim']])

### Optimisation

**Optimization experiment:** Evaluating recommendation diversity and coverage across weight configurations
- Metric 1: Intra-List Diversity (ILD) - average pairwise distance among top N recommendations
- Metric 2: Catalog Coverage - proportion of unique members ever recommended across a sample of 200 queries
- Metric 3: Cold-Start / Popularity Gini - measuring whether recommendations concentrate on only a few popular members


In [ ]:
# Store target and recommendation by index directly in recommendation function
class CyNamRecommenderOptimized:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.tfidf_cols = [c for c in data.columns if c.startswith('tfidf_')]
        self.sector_cols = [c for c in data.columns if c.startswith('sec_')]
        self.X_tfidf = self.data[self.tfidf_cols].values
        self.X_sector = self.data[self.sector_cols].values
        self.X_sen = self.data['norm_seniority'].values
        self.orgs = self.data['organization'].fillna(
            '').str.strip().str.lower().values

    def get_recommendation_indices(self, target_idx, weights=(0.5, 0.3, 0.2), top_n=5, mode='homophily'):
        w_int, w_sec, w_sen = weights
        target_sen = self.X_sen[target_idx]

        sim_interest = cosine_similarity(
            self.X_tfidf[target_idx:target_idx+1], self.X_tfidf).flatten()
        sim_sector = cosine_similarity(
            self.X_sector[target_idx:target_idx+1], self.X_sector).flatten()

        if mode == 'homophily':
            sen_score = 1.0 - np.abs(self.X_sen - target_sen)
        elif mode == 'mentorship':
            if target_sen <= 0.5:
                sen_score = np.maximum(0, self.X_sen - target_sen)
            else:
                sen_score = np.maximum(0, target_sen - self.X_sen)
        else:
            sen_score = 1.0 - np.abs(self.X_sen - target_sen)

        final_scores = (w_int * sim_interest) + \
            (w_sec * sim_sector) + (w_sen * sen_score)

        # In-company suppression
        target_org = self.orgs[target_idx]
        if target_org:
            final_scores[self.orgs == target_org] = -1.0
        else:
            final_scores[target_idx] = -1.0

        top_indices = np.argsort(final_scores)[::-1][:top_n]
        return top_indices, final_scores[top_indices]


opt_engine = CyNamRecommenderOptimized(df)

In [ ]:
# Evaluate metrics robustly
def evaluate_grid(opt_engine, sample_indices, weights, top_n=5):
    recommended_indices = set()
    all_recs = []
    ild_scores = []

    for idx in sample_indices:
        rec_idxs, scores = opt_engine.get_recommendation_indices(
            idx, weights=weights, top_n=top_n)
        recommended_indices.update(rec_idxs)
        all_recs.extend(rec_idxs)

        vecs = opt_engine.X_tfidf[rec_idxs]
        if len(vecs) > 1:
            # Cosine distance
            dists = pdist(vecs, metric='cosine')
            dists = np.nan_to_num(dists, nan=0.0)
            ild_scores.append(np.mean(dists))

    catalog_coverage = (len(recommended_indices) / len(opt_engine.data)) * 100
    avg_ild = np.mean(ild_scores) if ild_scores else 0.0

    # Rec frequency distribution (Gini coefficient of exposure)
    counts = pd.Series(all_recs).value_counts().values
    n = len(counts)
    gini = ((2 * np.sum(np.arange(1, n + 1) * np.sort(counts))) /
            (n * np.sum(counts)) - (n + 1) / n) if n > 1 else 0.0

    w_int, w_sec, w_sen = weights
    return {
        'Weights (Interest, Sector, Seniority)': f"({w_int:.2f}, {w_sec:.2f}, {w_sen:.2f})",
        'Intra-List Diversity (ILD ↑)': round(avg_ild, 4),
        'Catalog Coverage % (↑)': round(catalog_coverage, 2),
        'Exposure Gini Index (↓)': round(gini, 4)
    }

In [ ]:
# Run grid evaluation across sample of 150 diverse members
np.random.seed(42)
test_sample_indices = np.random.choice(len(df), size=200, replace=False)

weight_candidates = [
    (0.80, 0.10, 0.10),  # Heavy Interest Focus
    (0.50, 0.30, 0.20),  # Balanced Configuration
    (0.20, 0.60, 0.20),  # Heavy Sector Focus
    (0.20, 0.20, 0.60),  # Heavy Seniority Focus
    (0.34, 0.33, 0.33)  # Uniform Weights
]

opt_results = [evaluate_grid(opt_engine, test_sample_indices, w)
               for w in weight_candidates]
opt_df = pd.DataFrame(opt_results)
print(opt_df.to_string(index=False))